<a href="https://colab.research.google.com/github/zeyneppbaran/ml-proje/blob/main/notebooks/09_model_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 - Model: Random Forest

**Sorumlu:** Zeynep BARAN

**Ödevdeki karşılığı:** 4. Makine Öğrenmesi Modelleri

In [1]:
# KURULUM: bu hucreyi her oturumun basinda bir kez calistir.
# Repoyu Colab'e indirir; boylece src/ortak.py kullanilabilir.
import os
REPO_URL = "https://github.com/zeyneppbaran/ml-proje.git"  # TODO: kendi repo adresiniz
if not os.path.exists("/content/ml-proje"):
    !git clone {REPO_URL} /content/ml-proje
else:
    !git -C /content/ml-proje pull
%cd /content/ml-proje

Cloning into '/content/ml-proje'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 192 (delta 102), reused 51 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 3.30 MiB | 11.46 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/ml-proje


In [2]:
%pip install -q imbalanced-learn

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    cross_validate,
    RandomizedSearchCV,
    learning_curve,
    validation_curve
)

from src.ortak import (
    veri_yukle,
    veriyi_bol,
    ortak_cv,
    capraz_dogrula,
    degerlendir,
    RANDOM_STATE
)

from src.pipeline import (
    build_full_pipeline,
    build_smote_pipeline
)

df = veri_yukle()
X_train, X_test, y_train, y_test = veriyi_bol(df)

print("Eğitim veri boyutu:", X_train.shape)
print("Test veri boyutu:",X_test.shape)

print("\nEğitim sınıf dağılımı:")
print(y_train.value_counts())

print("\nEğitim sınıf oranları:")
print(y_train.value_counts(normalize=True))

Eğitim veri boyutu: (32950, 20)
Test veri boyutu: (8238, 20)

Eğitim sınıf dağılımı:
y
0    29238
1     3712
Name: count, dtype: int64

Eğitim sınıf oranları:
y
0    0.887344
1    0.112656
Name: proportion, dtype: float64


## E1. Temel (baseline) model: varsayılan ayarlarla sonuç

In [4]:
baseline_rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1
)

baseline_pipeline = build_full_pipeline(baseline_rf)

print(baseline_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed',
                                                   'daha_once_aranmis']),
                                                 ('mode',
                                                  Pipeline(

In [5]:
import time

baslangic = time.perf_counter()

baseline_cv = cross_validate(
    estimator=baseline_pipeline,
    X=X_train,
    y=y_train,
    cv=ortak_cv(),
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    return_train_score=True,
    n_jobs=-1,
    error_score="raise"
)

sure = time.perf_counter() - baslangic

print(f"Toplam CV süresi: {sure:.2f} saniye")

print("\nValidation sonuçları:")

for metrik in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    skorlar = baseline_cv[f"test_{metrik}"]

    print(
        f"{metrik:12s}: "
        f"{skorlar.mean():.4f} "
        f"(+/- {skorlar.std():.4f})"
    )

print("\nFold bazında F1 sonuçları:")

fold_sonuclari = pd.DataFrame({
    "Fold": range(1, 6),
    "Train F1": baseline_cv["train_f1"],
    "Validation F1": baseline_cv["test_f1"]
})

display(fold_sonuclari)

print("\nOrtalama Train F1:",
      baseline_cv["train_f1"].mean())

print("Ortalama Validation F1:",
      baseline_cv["test_f1"].mean())

Toplam CV süresi: 29.97 saniye

Validation sonuçları:
accuracy    : 0.8916 (+/- 0.0029)
precision   : 0.5371 (+/- 0.0281)
recall      : 0.2810 (+/- 0.0140)
f1          : 0.3685 (+/- 0.0135)
roc_auc     : 0.7695 (+/- 0.0074)

Fold bazında F1 sonuçları:


,Fold,Train F1,Validation F1
0,1,0.980365,0.374656
1,2,0.982492,0.375000
2,3,0.981905,0.350442
3,4,0.980881,0.386817
4,5,0.982606,0.355675



Ortalama Train F1: 0.9816497928592796
Ortalama Validation F1: 0.3685179667907665


**Yorum:** Random Forest modeli, varsayılan hiperparametreler ve 100 karar ağacı kullanılarak eğitilmiştir. Bu aşamada sınıf ağırlıklandırması veya SMOTE uygulanmamıştır. Model performansı 5 katlı Stratified Cross Validation yöntemiyle değerlendirilmiştir.

Ortalama validation F1 skoru 0.3685, accuracy değeri 0.8916 ve ROC-AUC değeri 0.7695 olarak elde edilmiştir. Recall değerinin 0.2810 seviyesinde kalması, modelin azınlık sınıfındaki müşterileri tespit etmekte zorlandığını göstermektedir.

Ortalama eğitim F1 skoru 0.9816 iken validation F1 skorunun 0.3685 olması, modelin eğitim verisine aşırı uyum sağladığına işaret etmektedir.

Sonraki aşamada sınıf ağırlıklandırması ve SMOTE yöntemleri karşılaştırılacak, hiperparametre optimizasyonu ile modelin genelleme performansının artırılması hedeflenecektir.

Nihai test seti bu aşamada kullanılmamıştır. Toplam cross validation süresi 29.97 saniye olarak ölçülmüştür.

## E2. Hiperparametre araması (GridSearchCV / RandomizedSearchCV + StratifiedKFold)

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## E3. Veri bölme yöntemi: hold-out + cross validation açıklaması

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## E4. Overfitting analizi (eğitim/doğrulama skoru, learning curve) ve alınan önlem

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## E5. Test seti sonuçları: confusion matrix, precision, recall, F1, ROC-AUC, ROC eğrisi

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## E6. Bu modelin avantajları ve dezavantajları

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)